# ViziGenesis V2 — Calibration & Backtest Analysis

This notebook demonstrates:
1. Walk-forward cross-validation results
2. Probability calibration (reliability diagrams)
3. Backtest equity curves with stress tests

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Simulate Walk-Forward Results

If you have trained a V2 model, load the saved metrics.
Otherwise we demonstrate with synthetic data.

In [ ]:
from backend.v2.calibration import (
    CombinedCalibrator, compute_classification_metrics,
    compute_reliability_diagram, detect_concept_drift,
)

# Synthetic validation predictions (replace with actual after training)
np.random.seed(42)
n = 500
# Simulate a model that's slightly miscalibrated (overconfident)
true_p = np.random.rand(n)
labels = (np.random.rand(n) < true_p).astype(int)
raw_probs = np.clip(true_p * 1.3 - 0.15, 0.01, 0.99)  # overconfident model

print(f"Samples: {n}, Positive rate: {labels.mean():.2%}")

## 2. Calibration

In [ ]:
# Fit calibrator on first 70%, evaluate on last 30%
split = int(0.7 * n)
cal = CombinedCalibrator()
cal.fit(raw_probs[:split], labels[:split])

# Calibrate test set
cal_probs = cal.predict(raw_probs[split:])
test_labels = labels[split:]

# Compare metrics
raw_metrics = compute_classification_metrics(raw_probs[split:], test_labels)
cal_metrics = compute_classification_metrics(cal_probs, test_labels)

print("RAW model:")
for k in ['accuracy', 'auc_roc', 'brier_score', 'log_loss']:
    print(f"  {k}: {raw_metrics[k]:.4f}")

print(f"\nCALIBRATED model (method={cal.best_method}):")
for k in ['accuracy', 'auc_roc', 'brier_score', 'log_loss']:
    print(f"  {k}: {cal_metrics[k]:.4f}")

In [ ]:
# Reliability diagram
raw_rd = compute_reliability_diagram(raw_probs[split:], test_labels)
cal_rd = compute_reliability_diagram(cal_probs, test_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, rd, title in [(axes[0], raw_rd, 'Raw'), (axes[1], cal_rd, 'Calibrated')]:
    bins = rd['bins']
    ax.bar(bins, rd['counts'], width=0.08, alpha=0.3, color='blue', label='Count')
    ax2 = ax.twinx()
    ax2.plot(bins, rd['true_probs'], 'ro-', label='Observed P(up)')
    ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax.set_xlabel('Predicted P(up)')
    ax.set_ylabel('Count')
    ax2.set_ylabel('Observed P(up)')
    ax.set_title(f'{title} Reliability Diagram')
    ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 3. Concept Drift Detection

In [ ]:
drift = detect_concept_drift(cal_probs, test_labels, threshold=0.30)
print(f"Drift detected: {drift['drift_detected']}")
print(f"Recent Brier: {drift['recent_brier']:.4f}")
print(f"Historical Brier: {drift['historical_brier']:.4f}")
print(f"Degradation: {drift['degradation']:.2%}")

## 4. Backtest

In [ ]:
from backend.v2.backtest import run_backtest, run_stress_tests

# Simulate a 2-year backtest
n_days = 500
np.random.seed(42)
dates = pd.bdate_range('2022-01-01', periods=n_days)
daily_returns = np.random.randn(n_days) * 0.015  # ~1.5% daily vol
direction_probs = np.clip(0.5 + daily_returns * 20 + np.random.randn(n_days) * 0.1, 0.05, 0.95)

result = run_backtest(dates, daily_returns, direction_probs)

print("=== Backtest Metrics ===")
for k, v in result.metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Equity curve
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# Top: equity curve
ax = axes[0]
eq = np.array(result.equity_curve)
ax.plot(eq, 'b-', linewidth=1.2, label='V2 Strategy')
# Buy-and-hold benchmark
bah = [result.equity_curve[0]]
for r in daily_returns:
    bah.append(bah[-1] * (1 + r))
ax.plot(bah, 'k--', linewidth=0.8, alpha=0.6, label='Buy & Hold')
ax.set_title('V2 Backtest Equity Curve')
ax.set_ylabel('Equity ($)')
ax.legend()

# Bottom: drawdown
ax2 = axes[1]
peak = np.maximum.accumulate(eq)
dd = (eq - peak) / peak * 100
ax2.fill_between(range(len(dd)), dd, 0, alpha=0.3, color='red')
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Trading Day')

plt.tight_layout()
plt.show()

In [ ]:
# Monthly PnL heatmap
monthly = result.monthly_pnl
if monthly:
    months = sorted(monthly.keys())
    vals = [monthly[m] for m in months]
    colors_m = ['green' if v > 0 else 'red' for v in vals]
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(months, vals, color=colors_m, alpha=0.7)
    ax.set_title('Monthly Returns (%)')
    ax.axhline(0, color='black', linewidth=0.5)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No monthly PnL data.')

## 5. Stress Tests

In [ ]:
stress = run_stress_tests(result, dates)
for name, data in stress.items():
    print(f"\n{name}:")
    for k, v in data.items():
        print(f"  {k}: {v}")